# PenjuruBus — Training Notebook

In [1]:
import os, json, joblib, warnings, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
warnings.filterwarnings('ignore')
ROOT = r'D:/penjurubus'
PROC_DIR = os.path.join(ROOT, 'data', 'processed_v3')
MODEL_DIR = os.path.join(ROOT, 'models')
OUT_DIR = os.path.join(ROOT, 'data', 'predictions_v4_search')
os.makedirs(MODEL_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
TARGET_COL = 'is_candidate_stop'

In [2]:
TRAIN_PATH = os.path.join(PROC_DIR, 'surabaya', 'features_v3.parquet')
VAL_PATH = os.path.join(PROC_DIR, 'yogyakarta', 'features_v3.parquet')
train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(VAL_PATH)
print(train_df.shape, val_df.shape)

(2165, 17) (320, 17)


In [3]:
def split_xy(df):
    y = df[TARGET_COL].astype(int)
    X = df.drop(columns=[TARGET_COL], errors='ignore')
    return X, y
X_train, y_train = split_xy(train_df)
X_val, y_val = split_xy(val_df)
drop_cols = [c for c in ['grid_id', 'city', 'label_method', 'geometry', 'centroid_x', 'centroid_y'] if c in X_train.columns]
X_train = X_train.drop(columns=drop_cols, errors='ignore')
X_val = X_val.drop(columns=drop_cols, errors='ignore')
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

In [4]:
for c in num_cols:
    med = X_train[c].median()
    X_train[c] = X_train[c].fillna(med)
    X_val[c] = X_val[c].fillna(med)
for c in cat_cols:
    mode = X_train[c].mode(dropna=True)
    fill = mode.iloc[0] if len(mode) else 'missing'
    X_train[c] = X_train[c].fillna(fill).astype(str)
    X_val[c] = X_val[c].fillna(fill).astype(str)
for c in num_cols:
    mu = X_train[c].mean(); sd = X_train[c].std(ddof=0); sd = 1.0 if pd.isna(sd) or sd == 0 else sd
    X_train[c] = (X_train[c] - mu) / sd
    X_val[c] = (X_val[c] - mu) / sd
for c in cat_cols:
    uniq = list(pd.Series(X_train[c].unique()).dropna())
    if len(uniq) <= 2:
        mapping = {v: i for i, v in enumerate(sorted(uniq))}
        X_train[c] = X_train[c].map(mapping).fillna(0).astype(float)
        X_val[c] = X_val[c].map(mapping).fillna(0).astype(float)
    else:
        X_train = X_train.drop(columns=[c])
        X_val = X_val.drop(columns=[c])
final_cols = X_train.columns.tolist()
X_val = X_val.reindex(columns=final_cols, fill_value=0)

In [5]:
models = {
    'logreg': LogisticRegression(max_iter=3000, class_weight='balanced', random_state=42),
    'rf': RandomForestClassifier(n_estimators=500, class_weight='balanced_subsample', random_state=42, n_jobs=-1),
    'gboost': GradientBoostingClassifier(random_state=42),
    'extra_trees': ExtraTreesClassifier(n_estimators=500, class_weight='balanced', random_state=42, n_jobs=-1),
    'svm': SVC(probability=True, class_weight='balanced', random_state=42),
}

def best_threshold(y_true, prob):
    grid = np.linspace(0.05, 0.95, 37)
    best_t, best_f1 = 0.5, -1
    for t in grid:
        pred = (prob >= t).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1; best_t = float(t)
    return best_t

def eval_thr(y_true, prob, thr):
    pred = (prob >= thr).astype(int)
    return {'accuracy': float(accuracy_score(y_true, pred)), 'precision': float(precision_score(y_true, pred, zero_division=0)), 'recall': float(recall_score(y_true, pred, zero_division=0)), 'f1': float(f1_score(y_true, pred, zero_division=0)), 'roc_auc': float(roc_auc_score(y_true, prob)) if len(np.unique(y_true)) > 1 else None, 'cm': confusion_matrix(y_true, pred).tolist()}

results = []
for name, model in models.items():
    print('[TRAIN]', name)
    model.fit(X_train, y_train)
    val_prob = model.predict_proba(X_val)[:, 1]
    thr = best_threshold(y_val, val_prob)
    val_eval = eval_thr(y_val, val_prob, thr)
    results.append({'name': name, 'model': model, 'threshold': thr, 'val_eval': val_eval, 'val_auc': val_eval['roc_auc']})
    print(val_eval)
best = max(results, key=lambda x: (x['val_auc'] if x['val_auc'] is not None else -1, x['val_eval']['f1']))
best_model = best['model']; best_thr = best['threshold']
print('BEST', best['name'], best_thr)

[TRAIN] logreg
{'accuracy': 0.8875, 'precision': 0.47058823529411764, 'recall': 1.0, 'f1': 0.64, 'roc_auc': 0.9996744791666667, 'cm': [[252, 36], [0, 32]]}
[TRAIN] rf
{'accuracy': 0.88125, 'precision': 0.45714285714285713, 'recall': 1.0, 'f1': 0.6274509803921569, 'roc_auc': 0.9306098090277777, 'cm': [[250, 38], [0, 32]]}
[TRAIN] gboost
{'accuracy': 0.89375, 'precision': 0.48484848484848486, 'recall': 1.0, 'f1': 0.6530612244897959, 'roc_auc': 0.9409722222222222, 'cm': [[254, 34], [0, 32]]}
[TRAIN] extra_trees
{'accuracy': 0.9375, 'precision': 0.65, 'recall': 0.8125, 'f1': 0.7222222222222222, 'roc_auc': 0.9788411458333334, 'cm': [[274, 14], [6, 26]]}
[TRAIN] svm
{'accuracy': 0.9, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'roc_auc': 0.5, 'cm': [[288, 0], [32, 0]]}
BEST logreg 0.95


In [6]:
joblib.dump({'model': best_model, 'threshold': best_thr, 'feature_cols': final_cols, 'num_cols': num_cols, 'cat_cols': cat_cols}, os.path.join(MODEL_DIR, 'step4_best_model.joblib'))
summary = {'best_model': best['name'], 'best_threshold': best_thr, 'validation_metrics': best['val_eval'], 'features': final_cols}
with open(os.path.join(MODEL_DIR, 'step4_best_summary.json'), 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
pd.DataFrame([{k: v for k, v in r.items() if k != 'model'} for r in results]).to_csv(os.path.join(OUT_DIR, 'model_search_summary.csv'), index=False)
print('saved')

saved


In [7]:
artifact = joblib.load(os.path.join(MODEL_DIR, 'step4_best_model.joblib'))
model = artifact['model']
cols = artifact['feature_cols']
if hasattr(model, 'feature_importances_'):
    imp = pd.DataFrame({'feature': cols, 'importance': model.feature_importances_}).sort_values('importance', ascending=False)
elif hasattr(model, 'coef_'):
    imp = pd.DataFrame({'feature': cols, 'importance': np.abs(model.coef_[0])}).sort_values('importance', ascending=False)
else:
    imp = pd.DataFrame({'feature': cols, 'importance': np.nan})
imp.to_csv(os.path.join(OUT_DIR, 'feature_importance.csv'), index=False)
imp.head(10)

,feature,importance
3,building_count,9.049736
2,poi_count,0.320161
6,road_density,0.133024
5,road_length,0.131525
0,area_m2,0.067306
7,dist_to_nearest_halte,0.014398
4,halte_count,0.006230
1,population,0.000000
8,pop_density,0.000000
9,pop_x_area,0.000000
